# BTIP Layer 5 — Time-Series Forecasting
Prophet + LSTM/TFT: 24h / 7-day P10/P50/P90 forecasts

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('dark_background')
CYAN, AMBER, RED, GREEN = '#00D4FF', '#FFB020', '#FF4444', '#00A86B'

FEATURE_STORE = '../data/processed/feature_store.parquet'
df = pl.read_parquet(FEATURE_STORE)
print(f'Loaded {len(df):,} rows, {df.width} columns')
print(df.columns)

## 1. Top-20 Junctions by Volume

In [ ]:
junction_col = 'junction_id_snapped' if 'junction_id_snapped' in df.columns else 'junction_name'
ts_col = 'created_datetime' if 'created_datetime' in df.columns else 'timestamp'

top20 = (
    df.group_by(junction_col)
    .len()
    .sort('len', descending=True)
    .head(20)
)
print(top20)

fig, ax = plt.subplots(figsize=(12, 5))
junctions = top20[junction_col].to_list()
counts = top20['len'].to_list()
bars = ax.barh(range(len(junctions)), counts, color=CYAN, alpha=0.8)
ax.set_yticks(range(len(junctions)))
ax.set_yticklabels([str(j)[:30] for j in junctions], fontsize=9)
ax.set_xlabel('Total Violations')
ax.set_title('Top-20 Junctions by Violation Volume (LSTM eligible)', color=CYAN)
plt.tight_layout()
plt.show()

## 2. Hourly Violation Time Series (Top Junction)

In [ ]:
top_junction = str(top20[junction_col][0])
print(f'Analysing junction: {top_junction}')

jdf = (
    df.filter(pl.col(junction_col).cast(pl.Utf8) == top_junction)
    .with_columns(pl.col(ts_col).cast(pl.Datetime).dt.truncate('1h').alias('hour_bucket'))
    .group_by('hour_bucket')
    .len()
    .sort('hour_bucket')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(jdf['hour_bucket'], jdf['len'], color=CYAN, linewidth=0.8, alpha=0.9)
ax.fill_between(jdf['hour_bucket'], jdf['len'], alpha=0.15, color=CYAN)
ax.set_title(f'Hourly Violations — Junction: {top_junction}', color=CYAN)
ax.set_xlabel('Date')
ax.set_ylabel('Violations per Hour')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

print(f'Total hours: {len(jdf)}, mean={jdf["len"].mean():.1f}, max={jdf["len"].max()}')

## 3. Prophet: Component Plots

In [ ]:
from backend.models.forecasting.prophet_forecast import _train_single

series = jdf.rename(columns={'hour_bucket': 'ds', 'len': 'y'})
series['is_rush_hour'] = series['ds'].dt.hour.isin([7,8,9,17,18,19,20]).astype(float)

print(f'Training Prophet for {top_junction} ({len(series)} hours)...')
model = _train_single(top_junction, series)

if model:
    future = model.make_future_dataframe(periods=168, freq='h')
    future['is_rush_hour'] = future['ds'].dt.hour.isin([7,8,9,17,18,19,20]).astype(float)
    forecast = model.predict(future)
    fig = model.plot_components(forecast)
    plt.suptitle(f'Prophet Components — {top_junction}', y=1.02, color=CYAN)
    plt.tight_layout()
    plt.show()
else:
    print('Prophet model returned None (insufficient data). Try a junction with more records.')

## 4. P10/P50/P90 Forecast Band

In [ ]:
from backend.models.forecasting.prophet_forecast import forecast as prophet_forecast

try:
    points = prophet_forecast(top_junction, horizon_hours=168)
    fdf = pd.DataFrame(points)
    fdf['ts'] = pd.to_datetime(fdf['ts'])

    fig, ax = plt.subplots(figsize=(16, 5))

    # Historical (last 14 days)
    hist = series.tail(336)
    ax.plot(hist['ds'], hist['y'], color='#666B7A', linewidth=0.8, label='Historical', alpha=0.8)

    # Forecast bands
    ax.fill_between(fdf['ts'], fdf['p10'], fdf['p90'], color=CYAN, alpha=0.15, label='P10–P90 band')
    ax.plot(fdf['ts'], fdf['p50'], color=CYAN, linewidth=2, label='P50 (median)')
    ax.plot(fdf['ts'], fdf['p10'], color=CYAN, linewidth=0.8, linestyle='--', alpha=0.6, label='P10')
    ax.plot(fdf['ts'], fdf['p90'], color=AMBER, linewidth=0.8, linestyle='--', alpha=0.6, label='P90')

    # 'Now' marker
    now = pd.Timestamp.now()
    ax.axvline(now, color=RED, linestyle=':', linewidth=1.5, label='Now')

    ax.set_title(f'7-Day P10/P50/P90 Forecast — {top_junction}', color=CYAN)
    ax.set_xlabel('Date')
    ax.set_ylabel('Violations / Hour')
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

    # Verify P10 <= P50 <= P90
    violations = ((fdf['p10'] > fdf['p50']) | (fdf['p50'] > fdf['p90'])).sum()
    print(f'Band ordering violations: {violations} (should be 0)')
except FileNotFoundError:
    print('Model not yet trained. Run: python scripts/train_forecasting.py')

## 5. Holiday Effect Visualisation

In [1]:
from backend.models.forecasting.prophet_forecast import BENGALURU_HOLIDAYS

print(f'Bengaluru holiday calendar: {len(BENGALURU_HOLIDAYS)} entries')
print(BENGALURU_HOLIDAYS.sort_values('ds').to_string(index=False))

if model:
    holiday_effect = forecast[forecast['ds'].isin(BENGALURU_HOLIDAYS['ds'].values)][['ds', 'yhat', 'holidays']]
    if not holiday_effect.empty:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.bar(range(len(holiday_effect)), holiday_effect['holidays'], color=AMBER, alpha=0.8)
        ax.set_xticks(range(len(holiday_effect)))
        ax.set_xticklabels([str(d)[:10] for d in holiday_effect['ds']], rotation=45, ha='right', fontsize=8)
        ax.set_title('Prophet Holiday Effect on Violation Forecast', color=AMBER)
        ax.set_ylabel('Effect on y (violations/hour)')
        ax.axhline(0, color='white', linewidth=0.5, alpha=0.5)
        plt.tight_layout()
        plt.show()
    else:
        print('No holiday overlap in forecast window — try a longer horizon.')

ModuleNotFoundError: No module named 'backend'

## 6. LSTM Training Curves (if trained)

In [ ]:
from pathlib import Path

if Path('../models/saved/lstm/lstm_model.pt').exists():
    print('LSTM model found. To retrain: python scripts/train_forecasting.py --skip-prophet --epochs 30')
    print('\nLSTM architecture:')
    print('  - Input: hourly violation counts, 14-day lookback (336 steps)')
    print('  - 2-layer LSTM, hidden_size=128, dropout=0.2')
    print('  - Output: 3 quantile heads (P10/P50/P90) × 168 steps (7d)')
    print('  - Loss: Pinball (quantile regression) at q=0.1, 0.5, 0.9')
    print('  - Optimizer: Adam, lr=1e-3, gradient clipping at 1.0')
    print('  - Best checkpoint saved by validation loss')
else:
    print('LSTM model not yet trained. Run: python scripts/train_forecasting.py')

## 7. Risk Calendar — 7×4 Shift Grid

In [ ]:
from backend.models.forecasting.prophet_forecast import risk_calendar

try:
    cal = risk_calendar(top_junction)
    days = [row[0]['day'] for row in cal]
    shifts = [cell['shift'] for cell in cal[0]]
    matrix = np.array([[cell['p50'] for cell in row] for row in cal])

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(4))
    ax.set_xticklabels(shifts)
    ax.set_yticks(range(7))
    ax.set_yticklabels(days)
    for i in range(7):
        for j in range(4):
            ax.text(j, i, f'{matrix[i,j]:.1f}', ha='center', va='center', fontsize=9,
                    color='black' if matrix[i,j] > matrix.max()*0.5 else 'white')
    plt.colorbar(im, ax=ax, label='P50 Violations/Hour')
    ax.set_title(f'7-Day Risk Calendar (P50) — {top_junction}', color=CYAN)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Train models first: python scripts/train_forecasting.py')

## 8. Prophet vs LSTM Comparison (top junction)

In [ ]:
from pathlib import Path as P

print('Model artifact inventory:')
print(f'  Prophet junction models: {len(list(P("../models/saved/prophet").glob("junction_*.pkl")))}')
print(f'  Prophet global model: {P("../models/saved/prophet/global_model.pkl").exists()}')
print(f'  LSTM model: {P("../models/saved/lstm/lstm_model.pt").exists()}')
print(f'  LSTM scaler: {P("../models/saved/lstm/scaler.pkl").exists()}')
print(f'  LSTM top-20 list: {P("../models/saved/lstm/top20_junctions.pkl").exists()}')

if P('../models/saved/lstm/top20_junctions.pkl').exists():
    import pickle
    with open('../models/saved/lstm/top20_junctions.pkl', 'rb') as f:
        top20_list = pickle.load(f)
    print(f'\nTop-20 junction IDs: {top20_list}')